In [0]:
dbutils.library.restartPython()

In [0]:
import os
import glob
import json
import urllib.request

from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import setup_catalog, setup_schema, setup_volume, setup_directory, read_and_write_raw

# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("dimension-ingestion")


# ============================================================
# 1. DEFINING CONFIG PATHS
# ============================================================
dbutils.widgets.text("path_dimension", "")
config_path_1 = dbutils.widgets.get("path_dimension")

dbutils.widgets.text("path_fact-01", "")
config_path_2 = dbutils.widgets.get("path_fact-01")

dbutils.widgets.text("path_fact-02", "")
config_path_3 = dbutils.widgets.get("path_fact-02")

dbutils.widgets.text("path_fact-03", "")
config_path_4 = dbutils.widgets.get("path_fact-03")

dbutils.widgets.text("path_fact-04", "")
config_path_5 = dbutils.widgets.get("path_fact-04")

dbutils.widgets.text("path_fact-05", "")
config_path_6 = dbutils.widgets.get("path_fact-05")

# ============================================================
# 2. COLLECT ALL CONFIG PATHS
# ============================================================
config_paths = [
    config_path_1,
    config_path_2,
    config_path_3,
    config_path_4,
    config_path_5,
    config_path_6,
]

# Filter out any empty/unset config paths
config_paths = [p for p in config_paths if p]

# ============================================================
# 3. RUN DATE (shared across all configs)
# ============================================================
run_date = date.today().isoformat()
logger.info("Load date is %s", run_date)

# ============================================================
# 4. PROCESS EACH CONFIG
# ============================================================
for config_path in config_paths:
    logger.info("Processing config: %s", config_path)

    with open(config_path, "r") as f:
        config = json.load(f)

    # ===============================================================
    # 5. READ DATA FROM CONFIG FILES
    # ===============================================================

    gdrive_file_ids = config["source"]["gdrive_file_ids"]

    catalog = config["target"]["catalog"]
    schema = config["target"]["schema"]
    volume = config["target"]["volume"]

    gdrive_base_url = config["source"]["gdrive_base_url"]
    base_path = config["target"]["base_path"]
    destination_folder = config["target"]["destination_folder"]

    logger.info("Reading data from %s", gdrive_base_url)

    # ============================================================
    # 6. OUTPUT PATH
    # ============================================================

    output_path = f"{base_path}/load_date={run_date}/{destination_folder}"
    logger.warning("Data will be stored in the path %s", output_path)

    # ============================================================
    # 7. CREATING A CATALOG, SCHEMA AND VOLUME IF DOES NOT EXIST
    # ============================================================

    logger.info("creating catalog if does not exist")
    setup_catalog(spark, catalog)

    logger.info("creating schema if does not exist")
    setup_schema(spark, catalog, schema)

    logger.info("creating volume if does not exist")
    setup_volume(spark, catalog, schema, volume)

    logger.info("Creating a directory in output_path %s", output_path)
    setup_directory(dbutils, output_path)

    # ===============================================================
    # 8. READ DATA FROM GOOGLE DRIVE AND WRITING TO DATABRICKS VOLUME
    # ===============================================================

    read_and_write_raw(urllib, gdrive_base_url, output_path, gdrive_file_ids)

logger.info("Files successfully succesfully ingested")

In [0]:
"""spark.sql("DROP VOLUME IF EXISTS employeedataplatform.bronze.raw_data")"""

In [0]:
"""%sql
DROP TABLE employeedataplatform.bronze.dimcancellationreason;
DROP TABLE employeedataplatform.bronze.dimcounty;
DROP TABLE employeedataplatform.bronze.dimpeople;
DROP TABLE employeedataplatform.bronze.dimshrinkagetype;"""